# Isolation Forest

## Setup

In [1]:
import sys
from pathlib import Path
import numpy as np
from sklearn.ensemble import IsolationForest

for _c in [Path.cwd(), Path.cwd() / "04_notebooks"]:
    if (_c / "shared_setup.py").exists():
        if str(_c) not in sys.path:
            sys.path.insert(0, str(_c))
        break

from shared_setup import load_and_prepare, evaluate, alarm_analysis

train_df, test_df, rul_df, useful_sensors, scaler = load_and_prepare()
X_train = train_df[useful_sensors].values
X_test  = test_df[useful_sensors].values

**Alarm-Logik in 3 Schritten:**
1. Anomaly-Score pro Zeitpunkt (Sensor-Vektor → Score)
2. Rolling Mean über `window` Zyklen zur Glättung
3. Z-Normalisierung relativ zu gesunden Zyklen → Alarm wenn `score_z < z_threshold`

In [2]:
# Parameter aus dem Gridsearch
n_estimators  = 100
contamination = 0.03
max_samples   = 128
window        = 10
z_threshold   = -2.5

# Schritt 1: Isolation Forest trainieren
model = IsolationForest(
    n_estimators=n_estimators,
    contamination=contamination,
    max_samples=max_samples,
    random_state=42,
)
model.fit(X_train)
train_df["anomaly_score"] = model.decision_function(X_train)

# Schritt 2: Rolling Mean glätten
train_df["score_rolling"] = train_df.groupby("unit_id")["anomaly_score"].transform(
    lambda x: x.rolling(10).mean()
)

In [3]:
# Schritt 3: Z-Normalisierung relativ zu gesunden Zyklen
healthy_filter = train_df["cycles"] <= train_df["max_cycle"] * 0.3
healthy_cycles = train_df[healthy_filter]

score_mean = healthy_cycles.groupby("unit_id")["score_rolling"].mean().reset_index()
score_mean.columns = ["unit_id", "score_mean"]
score_std  = healthy_cycles.groupby("unit_id")["score_rolling"].std().reset_index()
score_std.columns  = ["unit_id", "score_std"]

train_df = train_df.merge(score_mean, on="unit_id")
train_df = train_df.merge(score_std, on="unit_id")

train_df["score_normalized"] = (
    (train_df["score_rolling"] - train_df["score_mean"]) / train_df["score_std"]
).replace([np.inf, -np.inf], np.nan)

In [4]:
# Alarm: score_z < z_threshold
train_df["anomaly_flag"] = (train_df["score_normalized"] < z_threshold).fillna(False)

y_true = (train_df["RUL"] <= 30).astype(int)
y_pred = train_df["anomaly_flag"].astype(int)
evaluate(y_true, y_pred, "Trainingsdaten")

Trainingsdaten:
Precision: 0.5791
Recall:    0.8439
F1:        0.6869


In [5]:
# Wann wird Alarm ausgelöst? (gemessen am verbleibenden RUL)
first_alarm = (
    train_df[train_df["anomaly_flag"] == 1]
    .groupby("unit_id")["RUL"]
    .max()
)
print("Erster Alarm tritt auf bei RUL")
print(first_alarm.describe().round(1))

Erster Alarm tritt auf bei RUL
count    100.0
mean      62.7
std       54.6
min        5.0
25%       23.0
50%       47.5
75%       77.0
max      263.0
Name: RUL, dtype: float64


## Testphase: 100 unbekannte Triebwerke

- Triebwerke **MIT Alarm** → sollten niedrige RUL haben (bald ausgefallen)
- Triebwerke **OHNE Alarm** → sollten hohe RUL haben

In [6]:
# Anomaly Score auf Testdaten berechnen
test_df["anomaly_score"] = model.decision_function(X_test)

test_df["score_rolling"] = test_df.groupby("unit_id")["anomaly_score"].transform(
    lambda x: x.rolling(window).mean()
)

test_df = test_df.merge(score_mean, on="unit_id")
test_df = test_df.merge(score_std, on="unit_id")

test_df["score_normalized"] = (
    (test_df["score_rolling"] - test_df["score_mean"]) / test_df["score_std"]
).replace([np.inf, -np.inf], np.nan)

test_df["anomaly_flag"] = (test_df["score_normalized"] < z_threshold).fillna(False)

test_result = test_df.merge(rul_df, on="unit_id", how="left")
alarm_analysis(test_result, rul_df)

Units mit Alarm: 59 / 100

RUL-Verteilung (MIT Alarm):
count     59.0
mean      64.3
std       45.4
min        7.0
25%       20.0
50%       54.0
75%      107.0
max      145.0
Name: RUL, dtype: float64

RUL-Verteilung (OHNE Alarm):
count     41.0
mean      91.7
std       29.6
min       21.0
25%       77.0
50%       96.0
75%      113.0
max      137.0
Name: RUL, dtype: float64
